In [18]:
import os
from dotenv import load_dotenv
import fitz
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings,
)
load_dotenv()
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    is_separator_regex=False,
)

In [7]:
pdf_path = "attention.pdf"

pdf = fitz.open(pdf_path)
documents = []
for page_num, page in enumerate(pdf):
    text = page.get_text()
    if text.strip():
        documents.append(
            Document(
                page_content=text,
                metadata={
                    "source": os.path.basename(pdf_path),
                    "page": page_num + 1,
                },
            )
        )
pdf.close()
print(f"Loaded {len(documents)} pages.")

Loaded 11 pages.


In [9]:
chunks = text_splitter.split_documents(documents)
print(f"Total Chunks: {len(chunks)}")
print(chunks[0])

Total Chunks: 43
page_content='Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Experiments on two machine translation tasks show these models to
be superior in quality while being more parallelizable and requiring signiﬁcantly'

In [14]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
)

In [15]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="test",
)

In [16]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)

In [19]:
prompt = ChatPromptTemplate.from_template("""
You are a helpful AI assistant.

Answer the user's question ONLY using the provided context.
If the answer is not present in the context, reply:
"I couldn't find that information in the document."

Context:
{context}

Question:
{question}
""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

retrieval_chain = RunnableParallel({
    "context": retriever | format_docs,
    "question": RunnablePassthrough()
})

rag_chain = retrieval_chain | prompt | llm | StrOutputParser()

In [23]:
question = "what is transformer"
response = rag_chain.invoke(question)
print(response)

The Transformer is a model architecture that eschews recurrence and instead relies entirely on an attention mechanism to draw global dependencies between input and output. It allows for significantly more parallelization and follows an overall architecture using stacked self-attention and point-wise, fully connected layers for both the encoder and decoder.
